# Data Validation

Technical QA for the raw Contoso tables and the order-level analytical model. The key grain is **one row per `orderkey`**.

In [1]:
from pathlib import Path
import sys

def find_project_root():
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "sql").exists() and (candidate / "python").exists():
            return candidate
    raise FileNotFoundError("Project root not found")

PROJECT_ROOT = find_project_root()
sys.path.append(str(PROJECT_ROOT / "python" / "src"))

from db import get_engine
from sql_utils import read_sql_file

engine = get_engine()

## 1. Raw-data audit

Checks keys, quantities, prices, exchange rates, and customer/product foreign keys before any analytical aggregation.

In [2]:
raw = read_sql_file(engine, "validation/01_raw_data_audit.sql")
display(raw.T.rename(columns={0: "value"}))

zero_issue_cols = [
    "null_orderkey", "null_customerkey", "null_productkey", "null_orderdate",
    "invalid_quantity", "invalid_netprice", "invalid_unitcost", "invalid_exchangerate",
    "invalid_order_ownership", "duplicate_customerkeys", "duplicate_productkeys",
    "sales_rows_without_customer", "sales_rows_without_product",
]
assert (raw.loc[0, zero_issue_cols] == 0).all(), "Raw-data QA failed"

,value
sales_rows,199873
distinct_orders,83130
purchasing_customers,49487
min_order_date,2015-01-01
max_order_date,2024-04-20
null_orderkey,0
null_customerkey,0
null_productkey,0
null_orderdate,0
invalid_quantity,0


## 2. Order-grain reconciliation

The model must preserve every distinct raw `orderkey`, with no duplicated orders and no lost revenue or quantity.

In [3]:
orders = read_sql_file(engine, "validation/02_order_grain_audit.sql")
display(orders.T.rename(columns={0: "value"}))

r = orders.iloc[0]
assert r["fact_rows"] == r["fact_distinct_orders"] == r["raw_distinct_orders"]
assert r["fact_purchasing_customers"] == r["raw_purchasing_customers"]
assert r["duplicate_orderkeys"] == 0
assert r["null_business_keys"] == 0
assert r["negative_order_revenue"] == 0
assert r["non_positive_order_quantity"] == 0
assert abs(float(r["revenue_difference"])) < 0.01
assert r["quantity_difference"] == 0

,value
fact_rows,8.313000e+04
fact_distinct_orders,8.313000e+04
fact_purchasing_customers,4.948700e+04
fact_revenue_usd,2.062731e+08
fact_quantity,6.283700e+05
duplicate_orderkeys,0.000000e+00
null_business_keys,0.000000e+00
negative_order_revenue,0.000000e+00
non_positive_order_quantity,0.000000e+00
raw_distinct_orders,8.313000e+04


## 3. Customer and RFM sanity checks

Frequency must be based on actual orders. Tie checks verify that identical R/F/M inputs are not split across different scores.

In [4]:
rfm = read_sql_file(engine, "marts/05_rfm_segmentation.sql")

assert rfm["customerkey"].is_unique
assert len(rfm) == int(raw.loc[0, "purchasing_customers"])
assert rfm[["days_since_last_purchase", "frequency", "monetary_usd"]].notna().all().all()
assert (rfm["frequency"] >= 1).all()
assert rfm["r_score"].between(1, 5).all()
assert rfm["f_score"].between(1, 5).all()
assert rfm["m_score"].between(1, 5).all()

assert rfm.groupby("days_since_last_purchase")["r_score"].nunique().max() == 1
assert rfm.groupby("frequency")["f_score"].nunique().max() == 1
assert rfm.groupby("monetary_usd")["m_score"].nunique().max() == 1

display(rfm["frequency"].value_counts().sort_index().rename("customers").to_frame().head(10))

,customers
frequency,
1,27541
2,13914
3,5400
4,1873
5,559
6,145
7,39
8,14
9,1


## 4. Purchase-interval check

Intervals are calculated between consecutive **orders**, including zero-day intervals when a customer places multiple orders on the same date.

In [5]:
gaps = read_sql_file(engine, "marts/08_repeat_purchase_intervals.sql")
assert (gaps["gap_days"] >= 0).all()

median_gap = int(gaps["gap_days"].median())
print(f"Observed repeat-order intervals: {len(gaps):,}")
print(f"Median observed interval: {median_gap} days")
print("All validation checks passed.")

Observed repeat-order intervals: 33,643
Median observed interval: 557 days
All validation checks passed.
